# Test Set Evaluation

Runs all 18 checkpoints over the held-out test split. Results are written in long format (`model, seed, task, metric, value`) so per-seed values survive for the paired comparison in notebook 05 -- aggregates can be recomputed from them, but not the reverse. Raw test logits are saved per run so probabilities never need a second pass.

Validation joint accuracy is also recorded here: notebook 06 selects its Grad-CAM instance from it, keeping the test set out of that decision.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'
LOGITS_DIR = f'{RESULTS_DIR}/logits'

import os
for d in ['/content/data', LOGITS_DIR]:
    os.makedirs(d, exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

In [ ]:
import numpy as np
import pandas as pd
import torch

from split_utils import load_split
from evaluate import (
    load_single_task_model, load_flat24_model, load_multitask_model,
    predict_single_task, predict_flat24, predict_multitask,
    count_params, measure_inference_time_ms, save_test_logits,
)
from metrics import classification_metrics, evaluate_multitask
from comparison import to_long_format

train_df, val_df, test_df = load_split('/content/repo/02_Manifests/split_manifest.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43, 44]
len(test_df), DEVICE

In [ ]:
EXPERIMENTS = [
    {'name': 'ModelA_species',   'kind': 'single', 'task': 'species',   'num_classes': 8},
    {'name': 'ModelB_freshness', 'kind': 'single', 'task': 'freshness', 'num_classes': 3},
    {'name': 'ModelC_flat24',    'kind': 'flat24'},
    {'name': 'ModelD_EW',        'kind': 'mtl'},
    {'name': 'ModelD_UW',        'kind': 'mtl'},
    {'name': 'ModelD_DWA',       'kind': 'mtl'},
]

In [ ]:
def evaluate_run(exp, seed):
    """Returns one {model, seed, tasks: {...}} record plus the raw test logits."""
    run_name = f"{exp['name']}_seed{seed}"
    ckpt = f'{CHECKPOINT_DIR}/{run_name}.pt'
    tasks = {}

    if exp['kind'] == 'single':
        model = load_single_task_model(ckpt, exp['num_classes'], DEVICE)
        y_true, y_pred, logits = predict_single_task(model, test_df, DATASET_ROOT, exp['task'], DEVICE)
        tasks[exp['task']] = classification_metrics(y_true, y_pred, ordinal=(exp['task'] == 'freshness'))
    else:
        if exp['kind'] == 'flat24':
            model = load_flat24_model(ckpt, DEVICE)
            sp_t, sp_p, fr_t, fr_p, logits = predict_flat24(model, test_df, DATASET_ROOT, DEVICE)
        else:
            model = load_multitask_model(ckpt, DEVICE)
            sp_t, sp_p, fr_t, fr_p, logits = predict_multitask(model, test_df, DATASET_ROOT, DEVICE)
        report = evaluate_multitask(sp_t, sp_p, fr_t, fr_p)
        tasks['species'] = report['species']
        tasks['freshness'] = report['freshness']
        tasks['joint'] = {'joint_accuracy': report['joint_accuracy']}

    timing = measure_inference_time_ms(model, DEVICE)
    tasks['efficiency'] = {
        'params': count_params(model),
        'inference_ms': timing['mean_ms'],
        'inference_ms_std': timing['std_ms'],
    }
    return {'model': exp['name'], 'seed': seed, 'tasks': tasks}, logits, run_name

In [ ]:
records = []
for exp in EXPERIMENTS:
    for seed in SEEDS:
        record, logits, run_name = evaluate_run(exp, seed)
        save_test_logits(logits, run_name, LOGITS_DIR)
        records.append(record)
        print(f'[{run_name}] evaluated, logits {logits.shape}')

long_df = to_long_format(records)
long_df.to_csv(f'{RESULTS_DIR}/test_results_long.csv', index=False)
long_df.head(12)

## Readable summary

Derived from the long table for reading only. The long table stays the source of truth for notebook 05.

In [ ]:
headline = long_df[
    ((long_df.task == 'species') & (long_df.metric.isin(['accuracy', 'f1_macro'])))
    | ((long_df.task == 'freshness') & (long_df.metric.isin(['accuracy', 'f1_macro', 'qwk'])))
    | (long_df.task == 'joint')
]
headline.pivot_table(index='model', columns=['task', 'metric'], values='value',
                     aggfunc=['mean', 'std']).round(4)

## Validation joint accuracy (input to the Grad-CAM instance choice)

In [ ]:
val_rows = []
for exp in EXPERIMENTS:
    if exp['kind'] == 'single':
        continue
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt = f'{CHECKPOINT_DIR}/{run_name}.pt'
        loader = load_flat24_model if exp['kind'] == 'flat24' else load_multitask_model
        predict = predict_flat24 if exp['kind'] == 'flat24' else predict_multitask
        model = loader(ckpt, DEVICE)
        sp_t, sp_p, fr_t, fr_p, _ = predict(model, val_df, DATASET_ROOT, DEVICE)
        val_rows.append({'model': exp['name'], 'seed': seed,
                         'val_joint_accuracy': float(((sp_t == sp_p) & (fr_t == fr_p)).mean())})

val_joint_df = pd.DataFrame(val_rows)
val_joint_df.to_csv(f'{RESULTS_DIR}/validation_joint_accuracy.csv', index=False)
val_joint_df

## Confusion matrices

Per task, to see where errors sit -- whether the two *Oreochromis* species get confused, and whether freshness errors slip one level or jump two.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from metrics import task_confusion_matrix

CONFUSION_RUNS = {'ModelA_species': 42, 'ModelB_freshness': 42, 'ModelD_UW': 42}  # set from notebook 05
species_labels = sorted(test_df['species'].unique())
freshness_labels = ['Highly Fresh', 'Fresh', 'Not Fresh']

def plot_cm(cm, labels, ax, title):
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, ax=ax, cmap='Blues')
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')

for model_name, seed in CONFUSION_RUNS.items():
    exp = next(e for e in EXPERIMENTS if e['name'] == model_name)
    ckpt = f'{CHECKPOINT_DIR}/{model_name}_seed{seed}.pt'
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    if exp['kind'] == 'single':
        model = load_single_task_model(ckpt, exp['num_classes'], DEVICE)
        y_true, y_pred, _ = predict_single_task(model, test_df, DATASET_ROOT, exp['task'], DEVICE)
        labels = species_labels if exp['task'] == 'species' else freshness_labels
        plot_cm(task_confusion_matrix(y_true, y_pred), labels, axes[0], f'{model_name} -- {exp["task"]}')
        axes[1].axis('off')
    else:
        loader = load_flat24_model if exp['kind'] == 'flat24' else load_multitask_model
        predict = predict_flat24 if exp['kind'] == 'flat24' else predict_multitask
        model = loader(ckpt, DEVICE)
        sp_t, sp_p, fr_t, fr_p, _ = predict(model, test_df, DATASET_ROOT, DEVICE)
        plot_cm(task_confusion_matrix(sp_t, sp_p), species_labels, axes[0], f'{model_name} -- species')
        plot_cm(task_confusion_matrix(fr_t, fr_p), freshness_labels, axes[1], f'{model_name} -- freshness')

    plt.tight_layout()
    fig.savefig(f'{RESULTS_DIR}/{model_name}_seed{seed}_confusion.png', dpi=150)
    plt.show(); plt.close(fig)

Written to Drive: `test_results_long.csv`, `validation_joint_accuracy.csv`, the per-run logits, and the confusion matrices.